**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/jones-2021"], check=True)
    os.chdir("/content/encore/entries/jones-2021")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Aery Jones et al. (2021) from dandiset 000165: home cage (rest) sessions

**Paper.** Aery Jones EA, Rao A, Zilberter M, Djukic B, Bant JS, Gillespie AK, Koutsodendris N, Nelson M,
Yoon SY, Huang K, Yuan H, Gill TM, Huang Y, Frank LM (2021). Dentate gyrus and CA3 GABAergic
interneurons bidirectionally modulate signatures of internal and external drive to CA1. *Cell
Reports* 37, 110159. [doi:10.1016/j.celrep.2021.110159](https://doi.org/10.1016/j.celrep.2021.110159).
The claims file (`claims/jones-2021.yaml`) was written from the accepted author manuscript in PubMed
Central (PMC9069800) with its main supplement; Table S7 was not available to the reader.

**Dandiset.** [DANDI:000165](https://dandiarchive.org/dandiset/000165/0.211118.1526), version
0.211118.1526 (doi:10.48324/dandi.000165/0.211118.1526). The paper links the draft, not a version.
This is the only published version, and the draft has the same 572 assets and size, so the most
recent published version is pinned. Files are streamed with remfile and h5py; nothing is written to disk.

**Notebooks.** `rest_home_cage.ipynb` tests the claims on the home cage (rest) sessions;
`run_linear_track.ipynb` tests those on the linear track (run) sessions; `csd_cohort1.ipynb` tests
the CSD claim on the single-shank home cage sessions of cohort 1. Methods are in the helper
`jones2021.py`, whose docstring says where each parameter comes from.

**Common method (Method M of the claims file).** For each feature, a linear mixed model feature ~
treatment + (treatment|animal), fitted by maximum likelihood, with an F test of the treatment
coefficient (CNO minus vehicle); the model is fitted exactly from per-mouse, per-treatment sufficient
statistics (cell counts, means and sums of squares), which gives the same likelihood as fitting every
observation. The genotype test is the likelihood ratio of feature ~ treatment + (treatment|animal)
against feature ~ treatment:genotype + (treatment:genotype|animal) on the pooled data, with 9 degrees
of freedom as Table S3 gives. Two readings of the log transform are run: (a) log when the within-cell
residuals have skewness above 1 (assumption: the Methods log right-skewed features without a rule);
(b) log for every feature (the code). Significance is Holm-Bonferroni at 0.05 over the paper's family
(24 rest or 36 run comparisons per group or pair), with comparisons not computed here ranked after
those computed.

| Claim | Tested in | Status |
|---|---|---|
| fig1-tables1-expression | none | blocked: data_missing (histology not deposited) |
| figs1ef-exvivo-cno | none | blocked: data_missing (slice recordings not deposited) |
| fig2ef-csd-sr | csd | tested (cohort 1, `csd_cohort1.ipynb`) |
| fig3b-mua-during-swr | rest | tested |
| fig3c-swr-rate-bidirectional | rest | tested |
| fig3d-swr-chains | rest | tested |
| fig3e-swr-length-sst | rest | tested |
| swr-longer-100ms-sst | rest | tested |
| fig3f-swr-frequency-pv | rest | tested |
| fig3g-sw-amplitude-pv | rest | tested |
| figs2a-swr-size-pv | rest | tested |
| tables2-fast-ripples-pv | none | blocked: data_missing (LFP filtered at 300 Hz, sampled at 1 kHz) |
| tables2-detection-unchanged | rest | tested |
| tables2-rest-sg-pv | rest | tested |
| fig4d-swr-sg-power-pv | rest | tested |
| fig4e-swr-sg-coherence-pv | rest | tested |
| figs2cd-sg-predicts-sw-frequency | rest | tested |
| ca3-swr-overlap | rest | tested |
| figs3-ca3-swr-pv | rest | tested |
| figs3c-ca3-swr-rate-sst | rest | tested |
| figs3-dentate-spikes | rest | tested |
| fig5b-mua-run to figs5-theta-coherence-frequency (14 claims) | run | tested |
| running-speed-unchanged | run | tested |
| tables4-genotype-baseline | run | tested |
| tables6-sex-swr-size | rest and run | tested |
| sex-stratified-effects | rest and run | tested |
| figs6-double-like-pv, figs6-double-unlike-sst, figs6-empty-vector-control | rest and run | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import jones2021 as J

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
started = time.time()

assets = J.list_assets()
print(f"dandiset {J.DANDISET_ID} version {J.DANDISET_VERSION}: {len(assets)} files, "
      f"{assets['size'].sum() / 1e9:.1f} GB")
print(assets.kind.value_counts().to_string())
hc_assets = assets[assets.kind == "HomeCage"]
meta = hc_assets.merge(J.scan(hc_assets.asset_id.tolist(), workers=16), on="asset_id")
print("metadata read from", len(meta), "home cage files; errors:", int(meta.get("error", pd.Series(dtype=str)).notna().sum()),
      f"({time.time() - started:.0f} s)")

## Data check

The file metadata of every home cage file, before any selection.

In [ ]:
meta["group"] = [J.group_of(r) for _, r in meta.iterrows()]
meta["minutes"] = meta.n_samples / 1000 / 60
print("Sampling rate(s):", meta.rate.unique(), " LFP unit attribute:", meta.unit.unique())
print("Session descriptions:", meta.description.unique())
print("\nMice and files by probe, group and treatment:")
print(meta.groupby(["probe", "group", "treatment"]).agg(files=("path", "size"), mice=("subject_id", "nunique")).to_string())
print("\nSession length (min) by probe:")
print(meta.groupby("probe").minutes.describe().round(1).to_string())
seq = (meta.sort_values(["subject_id", "session_number"]).groupby("subject_id")
       .agg(group=("group", "first"), probe=("probe", "first"),
            sequence=("treatment", lambda t: "".join("C" if x == "CNO" else "v" for x in t)),
            sessions=("session_number", lambda s: ",".join(map(str, s))),
            minutes=("minutes", "median")))
print("\nTreatment sequence per mouse in session order (v vehicle, C CNO):")
print(seq.round(1).to_string())
dups = meta[meta.duplicated(["subject_id", "session_number", "n_samples", "n_spikes"], keep=False)]
print("\nFiles with the same mouse, session number, LFP length and spike count:")
print(dups[["path", "n_samples", "n_spikes", "start"]].to_string())

Checks of the metadata against the paper and the files:

- Every file has 32 sites at 1000 Hz with the LFP unit given as microvolts; the session description
  says "1 hour after CNO injection" for vehicle sessions too.
- Mice per group match the paper: 5 PV-Cre and 4 SST-Cre on the single-shank probe (cohort 1), and on
  the 4-shank probe 16 PV-Cre, 16 SST-Cre, 13 PV-Cre/SST-Cre and 8 empty vector mice (cohorts 2 and 3).
- Four 4-shank mice (Beverly and Worf, PV-Cre; Picard and Riker, SST-Cre) have only sessions labelled
  CNO, numbered 8 to 17. They give no vehicle sessions, so they are left out: 14 PV-Cre and 14 SST-Cre
  mice are tested where the paper has 16 each.
- Cohort 2 mice alternate vehicle and CNO, as the Methods say; cohort 3 mice (25 min home cage
  sessions) have three vehicle sessions followed by three CNO sessions. Treatment is then
  confounded with session order in cohort 3.
- Home cage sessions of cohort 3 last about 25 min (median by mouse in the table); the Methods give
  60 min home cage sessions. Many cohort 2 mice have 7 sessions where the Methods give 6.
- One session, Chakotay01, is deposited twice under two file names with identical LFP length, spike
  count and start time; it is counted once.

In [ ]:
both = seq[seq.sequence.str.contains("v") & seq.sequence.str.contains("C")].index
use = meta[meta.probe.str.contains("A4x8") & meta.subject_id.isin(both)]
use = use.drop_duplicates(["subject_id", "session_number", "n_samples", "n_spikes"])
print("Excluded mice (4-shank) without both treatments:",
      sorted(set(meta[meta.probe.str.contains("A4x8")].subject_id) - set(both)))
print("Files analysed:", len(use), "of", len(meta))
print(use.groupby(["group", "treatment"]).agg(files=("path", "size"), mice=("subject_id", "nunique")).to_string())
t0 = time.time()
results = J.process_many(use.asset_id.tolist(), "HomeCage", workers=4)
errors = [r for r in results if "error" in r]
print(f"processed {len(results)} files in {(time.time() - t0) / 60:.1f} min; failed: {len(errors)}")
for e in errors:
    print("  ", e)

In [ ]:
S = J.session_table(results)
S = S[S.animal.notna()] if "animal" in S else S
print("Sessions skipped:", S[S.skipped.notna()][["session", "skipped"]].to_string() if S.skipped.notna().any() else "none")
S = S[S.skipped.isna()].reset_index(drop=True)
chk = pd.DataFrame([dict(session=results[i]["meta"]["session_id"], group=g,
                         lfp_min=results[i]["lfp_range"][0], lfp_max=results[i]["lfp_range"][1],
                         conversion=results[i]["conv"], pos_hz=results[i]["pos"]["pos_hz"], lfp_t0=results[i]["pos"]["lfp_t0"],
                         pos_t0=results[i]["pos"]["pos_t0"],
                         pos_end_minus_lfp_end=results[i]["pos"]["pos_t1"] - results[i]["n_samples"] / 1000,
                         x_range=np.ptp(results[i]["pos"]["x_range"]), y_range=np.ptp(results[i]["pos"]["y_range"]),
                         rest_min=results[i]["rest_s"] / 60, swr=len(results[i]["swr"]),
                         artifacts=results[i]["n_artifact"],
                         ripple_mean=results[i]["ripple_band"]["mean"], ripple_sd=results[i]["ripple_band"]["sd"],
                         zero_spikes=results[i]["n_zero_spikes"], n_ca1_pyr=len(results[i]["regions"]["ca1_pyr"]),
                         n_ca1_sr=len(results[i]["regions"]["ca1_sr"]), n_ca3=len(results[i]["regions"]["ca3"]),
                         n_dg=len(results[i]["regions"]["dg"]),
                         ca3_swr=len(results[i].get("ca3_swr", [])), ds=len(results[i].get("ds", [])),
                         mua_lag=results[i]["mua_lag"], mua_lag_ratio=results[i]["mua_lag_peak_ratio"],
                         seconds=results[i]["seconds"], maxrss_gb=results[i]["maxrss_kb"] / 1e6)
                    for i, g in zip(S.i, S.group)])
print("Per-session values (ranges over sessions):")
print(chk.drop(columns=["session", "group"]).describe().round(3).T.to_string())
print("\nSessions whose LFP reaches the int16 limits (-32768 or 32767):",
      int(((chk.lfp_max >= 32767) | (chk.lfp_min <= -32768)).sum()), "of", len(chk))
print("Sessions without a 'ca1-pyr 1' site, CA3 or DG sites:",
      int((chk.n_ca1_pyr == 0).sum()), int((chk.n_ca3 == 0).sum()), int((chk.n_dg == 0).sum()))
print("\nSWRs per minute of rest by group and treatment (mean over sessions):")
chk["swr_per_min"] = chk.swr / chk.rest_min
print(chk.assign(treatment=S.treatment).groupby(["group", "treatment"]).swr_per_min.describe().round(2).to_string())
print("\nLFP starting_time (s) over sessions:", sorted(set(chk.lfp_t0)), "; first position timestamp (s):", sorted(set(chk.pos_t0.round(3)))[:5])
off = pd.DataFrame([dict(session=results[i]["meta"]["session_id"], **results[i]["sw_offset"][results[i]["regions"]["ca1_sr_ref"][0]])
                    for i in S.i if results[i]["regions"]["ca1_sr_ref"]])
print("At the SW site (CA1-sr on the SWR shank), per session: median raw LFP over rest and mean of the 0-30 Hz trace (file units):")
print(off[["median_lfp", "mean_sw"]].describe().round(1).T.to_string())


Checks of the data:

- The LFP reaches the int16 limits (-32768 or 32767) in some sessions. The ripple-band envelope mean
  over rest (ripple_mean above) is close to Table S2's 72.39 uV in file units, which is consistent with
  the file units being microvolts as the unit attribute says, with the extreme values clipped. The
  one claim with expected values in microvolts (tables2-rest-sg-pv) is still compared both in file
  units and scaled by 0.195 (the Intan step), since neither is shown.
- Every file gives the LFP a starting_time of 0 s, and position timestamps start at 0 s; the LFP is
  placed on the position clock from its starting_time.
- At the SW site the median raw LFP over rest and the mean of the 0-30 Hz trace are printed per
  session: the offsets are small next to the SW amplitudes (fig3g-sw-amplitude-pv).
- Position is in cm, at 23 to 32 Hz in the home cage, and ends within a fraction of a second of the
  LFP in most sessions (see pos_end_minus_lfp_end).
- Rest periods, SWR counts and the number of sites per region are printed per session; every session
  has a "ca1-pyr 1" site (the SWR site) and CA3 and DG sites.
- Every unit's spike train begins with a spike at exactly 0.0 s, which is dropped (zero_spikes is about 31
  per session).

In [ ]:
# Clock check of spikes against the LFP: CA1 MUA around CA1 SWR midpoints, per session
lag_ok = chk.mua_lag_ratio >= J.LAG_MIN_RATIO
print(f"CA1 MUA cross-correlogram with SWR midpoints, +-{J.LAG_MAX_S} s, 10 ms bins (3-bin smoothing):")
print(f"  sessions with a peak at least {J.LAG_MIN_RATIO} times the median: {lag_ok.sum()} of {len(chk)}")
print("  lag of the peak (s), MUA relative to SWR midpoint, those sessions:")
print(chk.mua_lag[lag_ok].describe().round(3).to_string())
print("  sessions with |lag| <= 20 ms:", int((chk.mua_lag[lag_ok].abs() <= 0.02).sum()),
      "; lag < -100 ms:", int((chk.mua_lag[lag_ok] < -0.1).sum()), "; lag > 100 ms:", int((chk.mua_lag[lag_ok] > 0.1).sum()))
sw = pd.concat([results[i]["swr"].assign(group=g) for i, g in zip(S.i, S.group)])
print("\nNormalised CA1 MUA during SWRs (SD units): as deposited mean", round(sw.mua_ca1.mean(), 3),
      "median", round(sw.mua_ca1.median(), 3), "; aligned mean", round(sw.mua_ca1_aligned.mean(), 3),
      "median", round(sw.mua_ca1_aligned.median(), 3), "(n SWRs", sw.mua_ca1_aligned.notna().sum(), ")")
print("Peak memory of the processing workers (GB):", round(chk.maxrss_gb.max(), 2),
      "; mean seconds per file:", round(chk.seconds.mean(), 1))

Clock of the spikes against the LFP. The paper verified SWR sites by "large increases in MUA during
SWRs". CA1 MUA has a clear peak around the SWRs detected on the LFP in most sessions, but the peak
is displaced from the SWR midpoint by a session-specific lag, mostly with the MUA earlier, up to about
2 s (summary above); only a few sessions have the peak within 20 ms. With the deposited times the mean
normalised CA1 MUA during SWRs is close to 0; with each session's spike times shifted by its lag it is
clearly positive. The spike times and the LFP are therefore not on the same clock as deposited. For
every measure of MUA locked to events (SWRs, CA3 SWRs, dentate spikes) the main reading uses the
shifted spike times, an assumption, and sessions without a clear peak are left out of it; results with
the deposited times are shown as variants. MUA rates over rest or run periods (seconds to minutes) are
used as deposited.

In [ ]:
# SWR-coincident SG power (fig4d): z-score each SWR's power by rest statistics (helper add_swr_sg_z)
USED = J.add_swr_sg_z(results, S)
print("CNO sessions:", len(USED), "; z-scored by an earlier vehicle session:", int(USED.earlier.sum()),
      "; by a later one (no earlier vehicle session):", int((~USED.earlier & USED.vehicle_session.notna()).sum()),
      "; without a vehicle session:", int(USED.vehicle_session.isna().sum()))
sgz = pd.concat([results[i]["swr"][[c for c in ("sg_ca1", "sg_ca3", "sg_dg", "sgcoh_ca3", "sgcoh_dg") if c in results[i]["swr"]]]
                 .assign(treatment=t) for i, t in zip(S.i, S.treatment)])
print("Per-SWR values by treatment (z units for power, coherence difference post minus pre):")
print(sgz.groupby("treatment").describe().T.round(3).to_string())


In [ ]:
GROUPS = ("PV", "SST", "PS", "EV")
PAIRS = (("PV", "SST"), ("PV", "PS"), ("SST", "PS"))
t0 = time.time()
TAB = J.build_tables(results, S, {**J.REST_FEATURES, **{k: g for k, (_, g) in J.REST_VARIANTS.items()}})
res = J.test_all(TAB, groups=GROUPS, pairs=PAIRS)
counter = {k: m for k, (m, _) in J.REST_VARIANTS.items()}
res = J.add_decisions(res, J.REST_FAMILY, list(J.REST_FEATURES), counter)
print(f"Rest comparisons computed: {len(J.REST_FEATURES)} features of the paper's {J.REST_FAMILY}; "
      f"tests run: {len(res)} ({time.time() - t0:.0f} s)")
print("Observations per feature (all groups):", {k: len(v) for k, v in TAB.items() if k in J.REST_FEATURES})
print(res[res.feature.isin(list(J.REST_FEATURES)) & (res.reading == "a")]
      .pivot_table(index="feature", columns="test", values="p", aggfunc="first").map(lambda v: f"{v:.2g}").to_string())

def part_table(parts):
    t = J.show_parts(res, parts)
    print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

## fig3b-mua-during-swr

> Suppressing either interneuron class increases the recruitment of DG and CA3 MUA to CA1 SWRs, but not of CA1 MUA.

Normalised MUA per SWR (code RTMUA.m normrate, as the criterion says) at each region's MUA site. The
main reading uses spike times shifted by the session's offset (data check); the deposited times are
shown below. Reading (a) logs features whose within-cell residuals are right skewed, (b) logs all.

In [ ]:
parts = [("mua_ca1", "PV", "ns"), ("mua_ca1", "SST", "ns"), ("mua_ca3", "PV", "up"), ("mua_ca3", "SST", "up"),
         ("mua_dg", "PV", "up"), ("mua_dg", "SST", "up")]
part_table(parts)
print("\nAs deposited (no clock correction):")
part_table([(f + "_deposited", g, e) for f, g, e in parts])

**Result: not reproduced.** With the spike times aligned to the SWRs, CA1 MUA recruitment does not change (PV-Cre p = 0.12, SST-Cre p = 0.30), as the claim says, but neither does CA3 MUA (p = 0.80 and 0.59) nor DG MUA (p = 0.09 and 0.09, the PV-Cre coefficient negative). With the deposited spike times no CA3 or DG increase is significant either. Both log readings give the same result. The outcome rests on the clock alignment, an assumption.

## fig3c-swr-rate-bidirectional

> Suppressing PV+ interneurons increases CA1 SWR rate while suppressing SST+ interneurons decreases it, and the two effects differ significantly.

SWR counts per complete 1 min bin of rest. The code's 5 SD events are a variant.

In [ ]:
part_table([("swr_rate", "PV", "up"), ("swr_rate", "SST", "down"), ("swr_rate", "PV vs SST", "sig")])
print("\n5 SD variant:")
part_table([("swr_rate_5sd", "PV", "up"), ("swr_rate_5sd", "SST", "down"), ("swr_rate_5sd", "PV vs SST", "sig")])
r = pd.DataFrame([dict(group=g, treatment=tr, animal=a, rate=np.mean(J.REST_FEATURES["swr_rate"](results[i])))
                  for i, g, tr, a in zip(S.i, S.group, S.treatment, S.animal)])
m = r.groupby(["group", "animal", "treatment"]).rate.mean().unstack()
print("\nSWRs per min, mean over mice of the mouse mean, and mice whose rate rose with CNO:")
print(pd.DataFrame({"vehicle": m.Vehicle.groupby(level=0).mean(), "CNO": m.CNO.groupby(level=0).mean(),
                    "rose": (m.CNO > m.Vehicle).groupby(level=0).sum(), "mice": m.dropna().groupby(level=0).size()}).round(2).to_string())

**Result: not reproduced.** Under reading (b), log for all features, SWR rate rises with PV+ suppression (coefficient 0.33 on the log scale, p = 1.0e-08), but under reading (a), where the rate is not logged, the PV-Cre rise (+4.2 per min, p = 0.014) does not pass Holm. SST+ suppression does not decrease the rate under either reading (p = 0.37 and 0.84), and the genotype test is not significant (p = 0.67 and 0.093). The 5 SD variant gives the same pattern. The empty vector mice also have a higher rate on CNO days (Summary table of the data check: 23.8 against 28.9 SWRs per min of rest), which points to the session order.

## fig3d-swr-chains

> Suppressing either interneuron class decreases the proportion of SWRs that occur in chains (within 200 ms of another SWR).

In [ ]:
part_table([("chain", "PV", "down"), ("chain", "SST", "down")])
print("\n5 SD variant:")
part_table([("chain_5sd", "PV", "down"), ("chain_5sd", "SST", "down")])

**Result: not reproduced.** The proportion of SWRs in chains does not decrease significantly with PV+ suppression (p = 0.48) and falls with SST+ suppression without passing Holm (p = 0.0064). With the code's 5 SD events the SST-Cre decrease is significant (p = 0.00044) and the PV-Cre effect is not (p = 0.35).

## fig3e-swr-length-sst

> Suppressing SST+ interneurons lengthens CA1 SWRs, an effect not seen with PV+ suppression, and the genotypes differ.

In [ ]:
part_table([("length", "SST", "up"), ("length", "PV", "ns"), ("length", "PV vs SST", "sig")])
print("\n5 SD variant:")
part_table([("length_5sd", "SST", "up"), ("length_5sd", "PV", "ns"), ("length_5sd", "PV vs SST", "sig")])

**Result: not reproduced.** SWRs lengthen with SST+ suppression on the log scale (coefficient 0.024, p = 0.011), which does not pass Holm; the PV-Cre effect is not significant (p = 0.072) and the genotype test is not (p = 0.30). The 5 SD variant gives p = 0.0057 for SST-Cre, also short of the threshold.

## swr-longer-100ms-sst

> With SST+ suppression the percentage of SWRs longer than 100 ms rises from 38.5% under vehicle to 49.6% under CNO.

Readings of the denominator: (a) the valid SWRs; (b) every detected event (before the edge and
artifact rules). Of pooling: (c) pooled over a treatment's sessions; (d) mean of session fractions.
The code's >= 100 ms is a variant. Expected 38.5 +/- 3.85 and 49.6 +/- 4.96 %.

In [ ]:
rows = []
for i, g, tr, a in zip(S.i, S.group, S.treatment, S.animal):
    for den, L in (("a valid", results[i]["swr"].length.values), ("b all detected", results[i]["all_event_lengths"])):
        for op, f in (("> 100 ms", lambda x: x > 0.1 + 1e-9), (">= 100 ms", lambda x: x >= 0.1 - 1e-9)):
            rows.append(dict(group=g, animal=a, treatment=tr, denominator=den, rule=op, n=len(L), long=int(f(L).sum())))
L = pd.DataFrame(rows)
out = []
for (den, op), d in L[L.group == "SST"].groupby(["denominator", "rule"]):
    pooled = d.groupby(["animal", "treatment"])[["long", "n"]].sum()
    pooled = (100 * pooled.long / pooled.n).unstack()
    sess = d.assign(f=100 * d.long / d.n).groupby(["animal", "treatment"]).f.mean().unstack()
    for pool, m in (("c pooled", pooled), ("d session mean", sess)):
        m = m.dropna()
        tt = stats.ttest_rel(m.CNO, m.Vehicle)
        out.append(dict(denominator=den, rule=op, pooling=pool, mice=len(m), vehicle=m.Vehicle.mean(), CNO=m.CNO.mean(),
                        t=tt.statistic, p=tt.pvalue,
                        within_tol=abs(m.Vehicle.mean() - 38.5) <= 3.85 and abs(m.CNO.mean() - 49.6) <= 4.96))
out = pd.DataFrame(out)
print(out.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPV-Cre, reading (a)(c), for reference:")
d = L[(L.group == "PV") & (L.denominator == "a valid") & (L.rule == "> 100 ms")].groupby(["animal", "treatment"])[["long", "n"]].sum()
d = (100 * d.long / d.n).unstack().dropna()
print(f"vehicle {d.Vehicle.mean():.1f}%  CNO {d.CNO.mean():.1f}%  paired t p = {stats.ttest_rel(d.CNO, d.Vehicle).pvalue:.3g}  ({len(d)} mice)")

**Result: not reproduced.** In SST-Cre mice the percentage of SWRs longer than 100 ms rises with CNO under every reading (paired t p from 0.030 to 0.078). With the valid SWRs (reading a) the CNO values (40.7 to 42.5%) are below the expected 49.6 +/- 4.96%. With every detected event and session means (b, d) both values are within tolerance (41.8 and 45.0%), but the paired t test gives p = 0.070, so no reading has both values in tolerance and p < 0.05. 14 SST-Cre mice are tested, not 16.

## fig3f-swr-frequency-pv

> PV+, but not SST+, interneuron suppression increases the instantaneous frequency of CA1 SWRs.

In [ ]:
part_table([("freq", "PV", "up"), ("freq", "SST", "ns")])
print("\nGenotype test, reported:")
part_table([("freq", "PV vs SST", "sig")])
print("\n5 SD variant:")
part_table([("freq_5sd", "PV", "up"), ("freq_5sd", "SST", "ns")])

**Result: reproduced.** SWR instantaneous frequency rises with PV+ suppression (+4.2 Hz, p = 4.5e-05 untransformed; p = 3.6e-05 logged) and does not change significantly with SST+ suppression (p = 0.063 and 0.061). The genotype test, reported only, is not significant (p = 0.97 and 0.56). The 5 SD variant agrees.

## fig3g-sw-amplitude-pv

> PV+, but not SST+, suppression increases sharp-wave amplitude in CA1, and the genotypes differ.

Reading (a) one CA1-sr site (on the shank of the SWR site); (b) every CA1-sr site, one value per SWR and site.

In [ ]:
part_table([("sw_amp", "PV", "up"), ("sw_amp", "SST", "ns"), ("sw_amp", "PV vs SST", "sig")])
print("\nReading (b), every CA1-sr site:")
part_table([("sw_amp_all_sr_sites", "PV", "up"), ("sw_amp_all_sr_sites", "SST", "ns"), ("sw_amp_all_sr_sites", "PV vs SST", "sig")])
print("\n5 SD variant:")
part_table([("sw_amp_5sd", "PV", "up"), ("sw_amp_5sd", "SST", "ns"), ("sw_amp_5sd", "PV vs SST", "sig")])
print("\nSW amplitude range over SWRs (file units):", round(sw.sw_amp.min(), 1), "to", round(sw.sw_amp.max(), 1),
      "; median", round(sw.sw_amp.median(), 1))
print("\nVariant: SW amplitude of the 0-30 Hz trace as filtered (no mean removed):")
part_table([("sw_amp_raw", "PV", "up"), ("sw_amp_raw", "SST", "ns"), ("sw_amp_raw", "PV vs SST", "sig")])
print("\nVariant: maximum deviation from the median of the 200 ms before the SWR:")
part_table([("sw_amp_base", "PV", "up"), ("sw_amp_base", "SST", "ns"), ("sw_amp_base", "PV vs SST", "sig")])
print("\nVehicle means over mice of each mouse's mean SW amplitude (file units), against Table S4's 375 (PV-Cre) and 385 (SST-Cre) uV:")
m = pd.concat([results[i]["swr"][["sw_amp", "sw_amp_raw", "sw_amp_base"]].assign(group=g, animal=a, treatment=t)
               for i, g, a, t in zip(S.i, S.group, S.animal, S.treatment) if "sw_amp" in results[i]["swr"]])
vm = m[m.treatment == "Vehicle"].groupby(["group", "animal"]).mean(numeric_only=True).groupby(level=0).mean()
print(vm.round(1).to_string())
print("Scaled by 0.195:"); print((vm * 0.195).round(1).to_string())


**Result: not reproduced.** SW amplitude rises with PV+ suppression (p = 7.8e-08), but it also rises with SST+ suppression (p = 0.00207, just below the first Holm threshold of 0.05/24 = 0.00208), and the genotype test is not significant (p = 0.53; p = 0.15 with 5 degrees of freedom). Every CA1-sr site (reading b, SST-Cre p = 0.0011), the 5 SD SWRs (p = 0.00016), the trace without its mean removed (p = 0.0020) and the deviation from the median of the 200 ms before the SWR (p = 0.0020) give the same pattern. The measure is not dominated by an offset: the median raw LFP at the SW site is 39 file units (data check) against SW amplitudes with a median of 3274, and removing the mean changes nothing. The vehicle means are 3246 (PV-Cre) and 2999 (SST-Cre) file units against Table S4's 375 and 385 uV, about 8 times larger, so the paper's SW measure differs from this one in a way the Methods do not explain.


## figs2a-swr-size-pv

> SWR size is smaller when PV+ interneurons are suppressed, despite no change in CA1 MUA recruitment.

In [ ]:
part_table([("size", "PV", "down")])
print("\nReported: SST-Cre, genotype test, and the 5 SD variant:")
part_table([("size", "SST", "ns"), ("size", "PV vs SST", "sig"), ("size_5sd", "PV", "down")])

**Result: reproduced.** SWR size falls with PV+ suppression (-0.42 SD, p = 2.1e-05; logged p = 0.00014) and does not change with SST+ suppression (p = 0.50). The 5 SD variant agrees (p = 2.5e-07). The claim's "despite no change in CA1 MUA recruitment" matches fig3b-mua-during-swr (PV-Cre p = 0.12).

## tables2-detection-unchanged

> The SWR rate changes are not due to differences in SWR detection (ripple band mean and SD) or in baseline MUA, which do not change with treatment in either genotype.

Per mouse and treatment, means over sessions; the test as Table S2's note (paired t when the
differences pass Shapiro-Wilk and the variances an F test, else Wilcoxon), p < 0.05. MUA at each
region's MUA site; rest MUA rates do not depend on the spike clock offset at this scale (100 ms bins
over periods of minutes).

In [ ]:
rows = []
for i, g, tr, a in zip(S.i, S.group, S.treatment, S.animal):
    r = results[i]
    rows.append(dict(group=g, animal=a, cno=int(tr == "CNO"), ripple_mean=r["ripple_band"]["mean"],
                     ripple_sd=r["ripple_band"]["sd"], mua_ca1=r.get("rest_mua_ca1"), mua_ca3=r.get("rest_mua_ca3"),
                     mua_dg=r.get("rest_mua_dg")))
D = pd.DataFrame(rows)
out = []
for g in ("PV", "SST"):
    for v in ("ripple_mean", "ripple_sd", "mua_ca1", "mua_ca3", "mua_dg"):
        m = D[D.group == g].groupby(["animal", "cno"])[v].mean().unstack()
        out.append(dict(group=g, measure=v, **J.paired(m)))
out = pd.DataFrame(out)
out["significant"] = out.p < 0.05
print(out.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Of the ten tests of the claim of absence, nine are not significant, but rest CA1 MUA in SST-Cre mice is: the differences fail the normality or variance check, so the Table S2 rule takes the Wilcoxon test, which gives p = 0.049 (the paired t test gives p = 0.18; mean 13.0 against 8.2 spikes/s). The ripple band mean and SD do not change in either genotype (p from 0.22 to 0.74). The one failure is marginal and the effect is a decrease, not a change in detection.

## tables2-rest-sg-pv

> Suppressing PV+ interneurons raises SG power in CA1 across all rest and raises its variance in CA1, CA3 and DG; suppressing SST+ interneurons does not change baseline SG power.

Readings: (a) squared Hilbert envelope of the 30-50 Hz filtered trace; (b) multitaper band power
(uV^2/Hz in file units). The expected values (109.51 and 127.6 uV^2) are compared under each reading
in file units and with the file values scaled by 0.195 uV per unit (see the data check).

In [ ]:
rows = []
for i, g, tr, a in zip(S.i, S.group, S.treatment, S.animal):
    r = results[i]
    for reading, key in (("a envelope^2", "rest_sg"), ("b multitaper", "rest_sg_mt")):
        for reg, v in r[key].items():
            rows.append(dict(group=g, animal=a, cno=int(tr == "CNO"), reading=reading, region=reg, mean=v["mean"], sd=v["sd"]))
D = pd.DataFrame(rows)
out = []
for reading in D.reading.unique():
    for g in ("PV", "SST"):
        for reg in ("ca1_sr", "ca3", "dg"):
            for v in ("mean", "sd"):
                m = D[(D.reading == reading) & (D.group == g) & (D.region == reg)].groupby(["animal", "cno"])[v].mean().unstack()
                out.append(dict(reading=reading, group=g, region=reg, measure=v, **J.paired(m)))
out = pd.DataFrame(out)
out["significant"] = out.p < 0.05
print(out.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
ca1 = out[(out.group == "PV") & (out.region == "ca1_sr") & (out.measure == "mean")]
for _, r in ca1.iterrows():
    for scale in (1.0, 0.195 ** 2):
        print(f"{r.reading}, scale {scale:.4f}: PV CA1-sr vehicle {r.veh * scale:.4g}, CNO {r.cno * scale:.4g} "
              f"(expected 109.51 +/- 10.95, 127.6 +/- 12.76)")

**Result: partially reproduced.** In PV-Cre mice SG power in CA1-sr rises across rest (Wilcoxon p = 0.0005 under both readings) and its SD rises in CA1-sr, CA3 and DG (p from 0.0002 to 0.02), as claimed. In SST-Cre mice baseline SG power does not change in CA1-sr (p = 0.61), but it falls in CA3 (p = 0.0023) and DG (p = 0.04), which the claim excludes. The expected values do not match under either reading: the multitaper power in file units scaled by 0.195^2 gives 130 and 192 against 109.5 +/- 11.0 and 127.6 +/- 12.8, and the other scalings are further off. 13 PV-Cre mice have a CA1-sr site.

## fig4d-swr-sg-power-pv

> Suppressing PV+ interneurons increases SWR-coincident SG power in CA1, CA3 and DG, while suppressing SST+ interneurons does not change it; the genotypes differ in CA1.

Per SWR and site, multitaper power (100 ms window from SWR onset, 3 tapers, time-bandwidth 2, an
assumption) averaged over 30-50 Hz, z-scored by the mean and SD of the same measure over the 100 ms
windows of rest: CNO sessions by the mouse's latest earlier vehicle session (Methods), vehicle
sessions by their own (claims file). A region's value is the mean over its sites (CA1-sr, CA3, DG).
Variants: every session z-scored by its own statistics (code), and the 5 SD SWRs (code).

In [ ]:
part_table([("sg_ca1", "PV", "up"), ("sg_ca3", "PV", "up"), ("sg_dg", "PV", "up"), ("sg_ca1", "SST", "ns"),
            ("sg_ca3", "SST", "ns"), ("sg_dg", "SST", "ns"), ("sg_ca1", "PV vs SST", "sig")])
print("\nVariant: each session z-scored by its own rest statistics (code):")
part_table([("sg_ca1_own", "PV", "up"), ("sg_ca3_own", "PV", "up"), ("sg_dg_own", "PV", "up"), ("sg_ca1_own", "SST", "ns"),
            ("sg_ca3_own", "SST", "ns"), ("sg_dg_own", "SST", "ns"), ("sg_ca1_own", "PV vs SST", "sig")])
print("\nVariant: 5 SD SWRs, CA1:")
part_table([("sg_ca1_5sd", "PV", "up"), ("sg_ca1_5sd", "SST", "ns"), ("sg_ca1_5sd", "PV vs SST", "sig")])
m = TAB["sg_ca1"].groupby(["group", "animal", "cno"]).value.mean().unstack()
print("\nCA1 SWR-coincident SG power (z), mean over mice; mice whose value rose with CNO:")
print(pd.DataFrame({"vehicle": m[0].groupby(level=0).mean(), "CNO": m[1].groupby(level=0).mean(),
                    "rose": (m[1] > m[0]).groupby(level=0).sum(), "mice": m.dropna().groupby(level=0).size()}).round(3).to_string())


**Result: not reproduced.** With CNO sessions z-scored by the mouse's previous vehicle session (Methods), SWR-coincident SG power rises with PV+ suppression in CA1 (coefficient 0.54 on the log scale, p = 1.3e-11, 13 of 13 mice) and CA3 (p = 0.00046) but not significantly in DG (p = 0.19). No SST-Cre effect is significant (CA1 p = 0.0095, CA3 0.23, DG 0.14), as claimed, but the CA1 genotype test is not significant (p = 0.11; p = 0.013 with 5 degrees of freedom, also short of Holm). The partial rule needs the CA1 genotype test. The empty vector mice also rise in CA1 (p = 1.6e-05, figs6 table below), which may come from z-scoring a CNO session by another session's statistics or from session order (every cohort 3 mouse had its vehicle sessions first); no output here separates the two; with every session z-scored by its own statistics (code) no PV-Cre effect is significant (CA1 coefficient -0.06, p = 0.044). The outcome also rests on the tapers (3, time-bandwidth 2), an assumption.


## fig4e-swr-sg-coherence-pv

> Only PV+ suppression enhances the increase in SG coherence between CA1 and CA3 and between CA1 and DG during SWRs.

Per SWR, multitaper SG (30-50 Hz) coherence between the CA1 SWR site and each CA3 or DG site in the
100 ms after onset minus that in the 100 ms from 400 ms before onset, averaged over the region's
pairs; SWRs with another SWR within the 400 ms before onset are left out (Methods, paragraph 3).

In [ ]:
part_table([("sgcoh_ca3", "PV", "up"), ("sgcoh_dg", "PV", "up"), ("sgcoh_ca3", "SST", "ns"), ("sgcoh_dg", "SST", "ns"),
            ("sgcoh_ca3", "PV vs SST", "sig"), ("sgcoh_dg", "PV vs SST", "sig")])
n_all = sum(len(results[i]["swr"]) for i in S.i)
n_ok = sum(int(results[i]["swr"]["sgcoh_ca3"].notna().sum()) for i in S.i if "sgcoh_ca3" in results[i]["swr"])
print(f"\nSWRs with a clean 400 ms before onset: {n_ok} of {n_all}")
m = TAB["sgcoh_ca3"].groupby(["group", "cno"]).value.mean().unstack()
print("Mean coherence change (post minus pre), CA1-CA3, by group and treatment:"); print(m.round(4).to_string())
m = TAB["sgcoh_dg"].groupby(["group", "cno"]).value.mean().unstack()
print("Mean coherence change, CA1-DG:"); print(m.round(4).to_string())


**Result: not reproduced.** The SWR-related change in SG coherence does not rise with PV+ suppression, between CA1 and CA3 (coefficient -0.0013, p = 0.82) or CA1 and DG (+0.0006, p = 0.88). SST-Cre effects are not significant (p = 0.038 and 0.082), as claimed, and neither genotype test is (p = 0.077 and 0.60). 207,869 of 295,070 SWRs have no other SWR in the 400 ms before onset. The mean change, post minus pre, lies between -0.028 and -0.001 in every group and treatment.


## figs2cd-sg-predicts-sw-frequency

> Across PV-Cre mice, the CNO-induced change in CA1 SWR-coincident SG power predicts the change in SW amplitude and the change in SWR frequency.

Per PV-Cre mouse, CNO minus vehicle of CA1 SWR-coincident SG power (fig4d), SW amplitude (fig3g) and
SWR instantaneous frequency (fig3f): (a) difference of the mouse's means; (b) difference of the LMM
fitted conditional responses. Pearson correlation across mice. Expected R2 0.55 and 0.44.

In [ ]:
rows = []
for other, label in (("sw_amp", "SW amplitude"), ("freq", "SWR frequency")):
    for reading, fn in (("a mean difference", J.mean_changes), ("b LMM conditional", J.blup_changes)):
        x, y = fn(TAB["sg_ca1"], "PV"), fn(TAB[other], "PV")
        x, y = x.align(y, join="inner")
        r, p = stats.pearsonr(x, y)
        rows.append(dict(against=label, reading=reading, mice=len(x), r=r, R2=r * r, p=p,
                         sg_change_range=f"{x.min():.3g} to {x.max():.3g}", other_change_range=f"{y.min():.3g} to {y.max():.3g}"))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))


**Result: not reproduced.** Across 13 PV-Cre mice the change in CA1 SWR-coincident SG power correlates positively but not significantly with the change in SW amplitude (r = 0.36, R2 = 0.13, p = 0.23) and with the change in SWR frequency (r = 0.42, R2 = 0.18, p = 0.15), under both readings of a mouse's change; the expected R2 are 0.55 and 0.44. One PV-Cre mouse has no CA1-sr site, and the paper has 16 mice.


## ca3-swr-overlap

> Only a subset of CA3 SWRs, 37.18% averaged over vehicle sessions of PV-Cre and SST-Cre mice, overlap in time with CA1 SWRs.

A CA3 SWR overlaps when a valid CA1 SWR midpoint falls within it. Readings: (a) mean over sessions; (b) pooled over events.

In [ ]:
rows = []
for i, g, tr in zip(S.i, S.group, S.treatment):
    c = results[i].get("ca3_swr")
    if c is not None and g in ("PV", "SST") and tr == "Vehicle":
        rows.append(dict(group=g, n=len(c), overlap=int(c.overlap.sum())))
O = pd.DataFrame(rows)
O = O[O.n > 0]
a = (100 * O.overlap / O.n).mean()
b = 100 * O.overlap.sum() / O.n.sum()
print(f"Vehicle sessions: {len(O)}; CA3 SWRs {O.n.sum()}")
print(f"(a) mean over sessions {a:.2f}%  (b) pooled {b:.2f}%  expected 37.18 +/- 3.72")
print("Range of session percentages:", round((100 * O.overlap / O.n).min(), 1), "to", round((100 * O.overlap / O.n).max(), 1))

**Result: not reproduced.** 25.3% of CA3 SWRs overlap a CA1 SWR averaged over vehicle sessions (25.0% pooled over events), against 37.18 +/- 3.72%. The session values range from 5.8% to 49.4%.

## figs3-ca3-swr-pv

> Suppressing either class increases CA3 MUA recruitment to CA3 SWRs and lengthens CA3 SWRs; PV+ suppression decreases CA3 SWR rate and CA3 SWR instantaneous frequency.

Readings: (a) all valid CA3 SWRs (Methods); (b) CA3 SWRs that hold a CA1 SWR midpoint (code). CA3 MUA uses the aligned spike times.

In [ ]:
parts = [("ca3_mua", "PV", "up"), ("ca3_mua", "SST", "up"), ("ca3_length", "PV", "up"), ("ca3_length", "SST", "up"),
         ("ca3_rate", "PV", "down"), ("ca3_freq", "PV", "down")]
print("Reading (a), all CA3 SWRs:")
part_table(parts)
print("\nReading (b), CA3 SWRs overlapping a CA1 SWR:")
part_table([(f + "_overlap", g, e) for f, g, e in parts])
print("\nCA3 MUA with spike times as deposited:")
part_table([("ca3_mua_deposited", "PV", "up"), ("ca3_mua_deposited", "SST", "up")])

**Result: not reproduced.** Under reading (a), all CA3 SWRs, only the PV-Cre decrease in CA3 SWR rate holds (p = 0.00019); CA3 MUA recruitment does not rise in either genotype (p = 0.76 and 0.68), CA3 SWR length does not rise significantly (p = 0.29 and 0.15), and the PV-Cre frequency decrease is not significant (p = 0.17): one of six tests. Under reading (b), CA3 SWRs overlapping a CA1 SWR, none holds. The MUA part rests on the clock alignment, an assumption.

## figs3c-ca3-swr-rate-sst

> Suppressing SST+ interneurons increases CA3 SWR rate.

In [ ]:
part_table([("ca3_rate", "SST", "up"), ("ca3_rate_overlap", "SST", "up")])

**Result: not reproduced.** CA3 SWR rate does not increase with SST+ suppression; the coefficient is negative under both readings of which CA3 SWRs enter (-1.1 per min, p = 0.051, and -0.60 per min, p = 0.039, untransformed), and neither passes Holm. This agrees with the legend's p = 0.77 and not with the text.

## figs3-dentate-spikes

> DG MUA during dentate spikes does not change; suppressing either class decreases dentate spike rate; PV+ suppression increases dentate spike length and decreases their amplitude, while SST+ suppression has no detectable effect on these.

In [ ]:
part_table([("ds_mua", "PV", "ns"), ("ds_mua", "SST", "ns"), ("ds_rate", "PV", "down"), ("ds_rate", "SST", "down"),
                ("ds_length", "PV", "up"), ("ds_amp", "PV", "down"), ("ds_length", "SST", "ns"), ("ds_amp", "SST", "ns")])
print("\nDS MUA with spike times as deposited:")
part_table([("ds_mua_deposited", "PV", "ns"), ("ds_mua_deposited", "SST", "ns")])
ds = pd.concat([results[i]["ds"].assign(rest_min=results[i]["rest_s"] / 60) for i in S.i if "ds" in results[i]])
rate = len(ds) / sum(results[i]["rest_s"] for i in S.i) 
print(f"\nDentate spikes detected: {len(ds)}, {rate:.2f} per second of rest; length {ds.length.min()*1000:.0f} to "
      f"{ds.length.max()*1000:.0f} ms; amplitude (z units) {ds.amp.min():.2f} to {ds.amp.max():.2f}")

**Result: not reproduced.** DG MUA during dentate spikes does not change (p = 0.97 and 0.26) and PV+ suppression lowers dentate spike amplitude (p = 3.4e-05), as claimed, and SST+ suppression leaves length and amplitude unchanged after correction. But dentate spike rate does not fall in either genotype, and PV+ suppression does not lengthen them: five of the eight parts hold, and the partial rule needs the rate decreases. The Methods' detection, as implemented here, finds 8.58 events per second of rest, close to Table S4's dentate spike rates of 9.18 (PV-Cre) and 8.45 Hz (SST-Cre), which supports this reading of the Methods. The amplitude is in z units, where Table S4 gives uV. The outcome rests on this reading of the Methods, an assumption.

## tables6-sex-swr-size (rest features)

> During vehicle treatment, females have slightly larger SWR sizes than males, with no other sex differences.

Vehicle sessions of every mouse of cohorts 2 and 3 analysed here; LMM feature ~ sex + (sex|animal),
F test. The run features are tested in the linear track notebook. Reading (a) of the correction is
Holm over the rest features separately (m = 22 as Table S6); (b) uncorrected p < 0.05.

In [ ]:
Sv = S[S.treatment == "Vehicle"]
rows = []
for name in J.REST_FEATURES:
    d = TAB[name][TAB[name].session.isin(Sv.session)]
    if not len(d):
        continue
    sk = J.within_skew(d.assign(cno=(d.sex == "F").astype(int)))
    for log in (sk > J.SKEW_LOG,):
        dd = d.assign(cno=(d.sex == "F").astype(int))
        if log:
            dd["value"] = J.log_values(dd.value)
        cs = J.cell_stats(dd, "value")
        r = J.treatment_test(cs)
        means = d.groupby(["sex", "animal"]).value.mean().groupby(level=0).mean()
        rows.append(dict(feature=name, log=log, female_minus_male=r["coef"], p=r["p"], mean_F=means.get("F"),
                         mean_M=means.get("M"), mice_F=d[d.sex == "F"].animal.nunique(), mice_M=d[d.sex == "M"].animal.nunique()))
SEX = pd.DataFrame(rows)
SEX["holm_22"] = J.holm(SEX.p.values, 22)
SEX["p<0.05"] = SEX.p < 0.05
print(SEX.sort_values("p").to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result (rest part): SWR size does not differ by sex** in vehicle sessions (female minus male -0.08 SD, p = 0.52). No rest feature passes Holm over 22; CA3 SWR length has the smallest p (0.033, uncorrected). The run features are in the linear track notebook.

## sex-stratified-effects (rest features)

> Interneuron suppression has effects of the same direction and magnitude in both sexes on most features; the exceptions are SW amplitude with SST+ suppression (down in females, up in males), FG power with SST+ suppression (larger rise in females), and DG and CA3 MUA during movement (larger rise in females with suppression of either or both classes).

Per group, the likelihood ratio test of feature ~ treatment + (treatment|animal) against the model
with treatment by sex, the same machinery as the genotype test with sex in place of genotype.

In [ ]:
rows = []
for g in ("PV", "SST", "PS"):
    for name in J.REST_FEATURES:
        d = TAB[name][TAB[name].group == g].copy()
        if not len(d) or d.sex.nunique() < 2:
            continue
        if J.within_skew(d) > J.SKEW_LOG:
            d["value"] = J.log_values(d.value)
        csF, csM = J.cell_stats(d[d.sex == "F"], "value"), J.cell_stats(d[d.sex == "M"], "value")
        if csF.animal.nunique() < 2 or csM.animal.nunique() < 2:
            continue
        gt = J.genotype_test(csF, csM)
        rows.append(dict(group=g, feature=name, coef_F=gt["coef_a"], coef_M=gt["coef_b"], p=gt["p_df9"],
                         mice_F=csF.animal.nunique(), mice_M=csM.animal.nunique()))
SXR = pd.DataFrame(rows)
SXR["holm"] = False
for g, d in SXR.groupby("group"):
    SXR.loc[d.index, "holm"] = J.holm(d.p.values, len(d))
print(SXR.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSST-Cre SW amplitude:", SXR[(SXR.group == "SST") & (SXR.feature == "sw_amp")].to_string(index=False))

**Result (rest part): no named exception holds.** The SST-Cre SW amplitude effect does not differ by sex (p = 0.40; 3 female and 11 male SST-Cre mice are tested here). One rest feature has a significant sex interaction after Holm, PV-Cre CA3 SWR length. The run features are in the linear track notebook.

## figs6 claims (rest features): double suppression and empty vector

> figs6-double-like-pv: Suppressing both PV+ and SST+ interneurons cannot be told apart from suppressing PV+ interneurons only, except for theta modulation of SG in DG and CA3, which double suppression decreases.
>
> figs6-double-unlike-sst: Suppressing both interneuron types has largely distinct effects from suppressing SST+ interneurons only; the exceptions are FG frequency and CA1 FG power, which follow the direction of SST+ suppression.
>
> figs6-empty-vector-control: In mice injected with an empty vector, CNO and vehicle epochs do not differ.

Rest features computed here (22 of the 24; the two CSD features exist only for cohort 1, which has no P/S or empty vector mice); the run features are in the linear track notebook.

In [ ]:
gpv = gsst = res[res.feature.isin(list(J.REST_FEATURES))]
tab = pd.DataFrame({
    "PS treatment coef": gpv[(gpv.test == "PS") & (gpv.reading == "a")].set_index("feature").coef,
    "PS treatment p": gpv[(gpv.test == "PS") & (gpv.reading == "a")].set_index("feature").p,
    "PS sig": gpv[(gpv.test == "PS") & (gpv.reading == "a")].set_index("feature").significant,
    "PV vs PS p": gpv[(gpv.test == "PV vs PS") & (gpv.reading == "a")].set_index("feature").p,
    "PV vs PS sig": gpv[(gpv.test == "PV vs PS") & (gpv.reading == "a")].set_index("feature").significant,
    "SST vs PS p": gsst[(gsst.test == "SST vs PS") & (gsst.reading == "a")].set_index("feature").p,
    "SST vs PS sig": gsst[(gsst.test == "SST vs PS") & (gsst.reading == "a")].set_index("feature").significant,
    "EV coef": res[(res.test == "EV") & (res.reading == "a") & res.feature.isin(list(J.REST_FEATURES))].set_index("feature").coef,
    "EV p": res[(res.test == "EV") & (res.reading == "a") & res.feature.isin(list(J.REST_FEATURES))].set_index("feature").p,
    "EV sig": res[(res.test == "EV") & (res.reading == "a") & res.feature.isin(list(J.REST_FEATURES))].set_index("feature").significant,
})
print("Reading (a):")
print(tab.to_string(float_format=lambda v: f"{v:.3g}"))
for name, d, col in (("PV vs PS", gpv, "PV vs PS"), ("SST vs PS", gsst, "SST vs PS")):
    b = d[(d.test == col) & (d.reading == "b")]
    print(f"Reading (b), {name}: significant in", int(b.significant.sum()), "of", len(b), ":", b[b.significant].feature.tolist())
b = res[(res.test == "EV") & (res.reading == "b") & res.feature.isin(list(J.REST_FEATURES))]
print("Reading (b), EV significant:", b[b.significant].feature.tolist())
print("Mice: PS", S[S.group == "PS"].animal.nunique(), " EV", S[S.group == "EV"].animal.nunique())

**Result (rest part).** Double suppression against PV-Cre: no rest feature differs (reading a; reading b finds dentate spike length). Double suppression against SST-Cre: none of the 22 rest features differs, including SW amplitude (p = 0.68), SWR frequency (p = 0.40) and CA1 SWR-coincident SG power (p = 0.41), which the claim needs to differ. Empty vector mice change with CNO in SWR rate, chains, SWR size and CA1 SWR-coincident SG power (p from 2.9e-07 to 0.0017), where the claim says no feature changes. The two CSD comparisons exist only for cohort 1, which has no double suppression or empty vector mice, so 22 of the paper's 24 rest features enter here.


## Summary

| Claim | Outcome |
|---|---|
| fig3b-mua-during-swr | not reproduced (assumed clock alignment) |
| fig3c-swr-rate-bidirectional | not reproduced |
| fig3d-swr-chains | not reproduced |
| fig3e-swr-length-sst | not reproduced |
| swr-longer-100ms-sst | not reproduced |
| fig3f-swr-frequency-pv | reproduced |
| fig3g-sw-amplitude-pv | not reproduced |
| figs2a-swr-size-pv | reproduced |
| tables2-detection-unchanged | not reproduced |
| tables2-rest-sg-pv | partially reproduced |
| ca3-swr-overlap | not reproduced |
| figs3-ca3-swr-pv | not reproduced (assumed clock alignment) |
| figs3c-ca3-swr-rate-sst | not reproduced |
| figs3-dentate-spikes | not reproduced (assumed detection) |
| tables6-sex-swr-size, sex-stratified-effects, figs6 claims | decided with the run notebook; see the attempt record |
| fig1-tables1-expression, figs1ef-exvivo-cno, tables2-fast-ripples-pv | blocked, data missing |
| fig4d-swr-sg-power-pv | not reproduced (assumed tapers and z-scoring) |
| fig4e-swr-sg-coherence-pv | not reproduced |
| figs2cd-sg-predicts-sw-frequency | not reproduced |
| fig2ef-csd-sr | not reproduced (csd_cohort1.ipynb) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: 14 PV-Cre and 14 SST-Cre mice in place of 16 each (four mice have no vehicle sessions); spike times shifted per session to the SWRs for event-locked MUA; MUA at the labelled MUA sites; SWR boundaries where the envelope crosses its mean; SWR-coincident SG with 3 tapers, CNO sessions z-scored by the previous vehicle session; SW amplitude with the trace's mean removed; the genotype test fitted from cell statistics; the log transform under two readings; Holm over the comparisons computed here.

What was found about the dataset: spike times are not on the LFP clock (session-specific offsets up to about 2 s); four mice have only CNO-labelled sessions; cohort 3 has vehicle sessions before CNO sessions, not alternating, and the empty vector mice change with CNO; cohort 3 home cage sessions last 25 min, not 60; one session is deposited twice; the LFP values reach the int16 limits in 123 of 311 sessions; the SW amplitude in file units is about 8 times Table S4's; every spike train starts with a spike at 0 s.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")